# Transmission Line Data Analysis (Fall 2026)

## Notebook Summary

The notebook analyzes transmission line fault data and tests ML models for fault classification.
- Loads feature data from `../data/processed/stage4_features.csv` and `../data/processed/stage4_features_grouped_split.csv`.
- Removes metadata and selected feature columns.
- Separates the target variable, `class_name`, from the input features. 
- Splits data into training, validation, test, and Out-of-distribution (ood)
- Trains and evaluates:
    - Softmax Logistic Regression
    - Random Forest
    - XGBoost
    - MLP neural network
    - TCN neural network

In [1]:
# Libraries
from sys import platform

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
# XGBoost
import xgboost as xgb
# mlp model
import torch.nn as nn
import torch.optim as optim
# tcn
import torch.nn.functional as F


In [2]:
### Data Loading

df = pd.read_csv("../data/processed/stage4_features.csv")

df.head()

,scenario_id,split,class_name,send_v_rms_a_v,send_v_rms_b_v,send_v_rms_c_v,send_i_rms_a_a,send_i_rms_b_a,send_i_rms_c_a,send_v_rms_ratio_a,...,receive_v_spectral_entropy_mean,receive_v_spectral_entropy_max,receive_v_fundamental_residual_ratio_mean,receive_v_crest_factor_mean,receive_v_flatline_fraction_mean,receive_i_spectral_entropy_mean,receive_i_spectral_entropy_max,receive_i_fundamental_residual_ratio_mean,receive_i_crest_factor_mean,receive_i_flatline_fraction_mean
0,stage4_006324,train,CA,116648.243328,132606.498764,117525.522755,2036.110915,226.360257,2063.679653,0.882780,...,0.115058,0.118373,0.055208,1.378418,0.010860,0.114468,0.116752,0.054456,1.608260,0.003342
1,stage4_000708,train,Healthy,132162.440207,132008.443957,131239.898341,174.127951,173.272842,173.452631,1.000069,...,0.113745,0.117383,0.013537,1.426633,0.001671,0.113762,0.116331,0.013120,1.424010,0.000835
2,stage4_003544,train,CG,131777.450133,132478.795791,77860.328031,235.813976,263.024186,2145.019828,0.999050,...,0.185925,0.330113,0.209468,1.618260,0.010025,0.220438,0.428927,0.057327,1.502808,0.001671
3,stage4_002495,validation,BG,131254.093122,80240.971916,130988.264764,327.626774,3477.002206,300.159245,0.998033,...,0.163333,0.260361,0.165770,1.637747,0.013367,0.164593,0.263828,0.044094,1.449196,0.006683
4,stage4_004132,train,AB,131671.968270,124426.084239,132221.749750,1327.319630,1300.746614,200.872834,0.990787,...,0.115309,0.117159,0.026249,1.438159,0.004177,0.114655,0.119182,0.029347,1.424801,0.002506


In [3]:
metadata = [
    "scenario_id", "split", "class_name"]

In [4]:
### Data Preprocessing

# Drop columns
cols_to_drop = ['scenario_id']
df.drop(columns=cols_to_drop, errors='ignore', inplace=True)

df.head()

,split,class_name,send_v_rms_a_v,send_v_rms_b_v,send_v_rms_c_v,send_i_rms_a_a,send_i_rms_b_a,send_i_rms_c_a,send_v_rms_ratio_a,send_v_rms_ratio_b,...,receive_v_spectral_entropy_mean,receive_v_spectral_entropy_max,receive_v_fundamental_residual_ratio_mean,receive_v_crest_factor_mean,receive_v_flatline_fraction_mean,receive_i_spectral_entropy_mean,receive_i_spectral_entropy_max,receive_i_fundamental_residual_ratio_mean,receive_i_crest_factor_mean,receive_i_flatline_fraction_mean
0,train,CA,116648.243328,132606.498764,117525.522755,2036.110915,226.360257,2063.679653,0.882780,1.003292,...,0.115058,0.118373,0.055208,1.378418,0.010860,0.114468,0.116752,0.054456,1.608260,0.003342
1,train,Healthy,132162.440207,132008.443957,131239.898341,174.127951,173.272842,173.452631,1.000069,0.999684,...,0.113745,0.117383,0.013537,1.426633,0.001671,0.113762,0.116331,0.013120,1.424010,0.000835
2,train,CG,131777.450133,132478.795791,77860.328031,235.813976,263.024186,2145.019828,0.999050,0.999888,...,0.185925,0.330113,0.209468,1.618260,0.010025,0.220438,0.428927,0.057327,1.502808,0.001671
3,validation,BG,131254.093122,80240.971916,130988.264764,327.626774,3477.002206,300.159245,0.998033,0.614567,...,0.163333,0.260361,0.165770,1.637747,0.013367,0.164593,0.263828,0.044094,1.449196,0.006683
4,train,AB,131671.968270,124426.084239,132221.749750,1327.319630,1300.746614,200.872834,0.990787,0.945380,...,0.115309,0.117159,0.026249,1.438159,0.004177,0.114655,0.119182,0.029347,1.424801,0.002506


In [5]:
# Data Analysis
# List features variance
X = df.drop(columns=['class_name', 'split', 
                     'send_v_rms_ratio_a', 'send_v_rms_ratio_b', 'send_v_rms_ratio_c', 
                     'send_i_rms_ratio_a', 'send_i_rms_ratio_b', 'send_i_rms_ratio_c', 
                     'send_power_factor', 'send_active_power_w', 'send_reactive_power_var',
                     'receive_v_rms_a_v', 'receive_v_rms_b_v', 'receive_v_rms_c_v', 
                     'receive_i_rms_a_a', 'receive_i_rms_b_a', 'receive_i_rms_c_a', 
                     'receive_v_rms_ratio_a', 'receive_v_rms_ratio_b', 'receive_v_rms_ratio_c', 
                     'receive_i_rms_ratio_a', 'receive_i_rms_ratio_b', 'receive_i_rms_ratio_c', 
                     'receive_v0_rms_v', 'receive_v1_rms_v', 'receive_v2_rms_v', 
                     'receive_i0_rms_a', 'receive_i1_rms_a', 'receive_i2_rms_a', 
                     'receive_v0_v1_ratio', 'receive_v2_v1_ratio', 'receive_i0_i1_ratio', 
                     'receive_i2_i1_ratio', 'receive_active_power_w', 'receive_reactive_power_var', 
                     'receive_power_factor', 'receive_z_magnitude_a_ohm', 'receive_z_magnitude_b_ohm', 
                     'receive_z_magnitude_c_ohm', 'receive_v_rms_imbalance_pct', 'receive_i_rms_imbalance_pct', 
                     'send_v_spectral_entropy_mean', 'send_v_spectral_entropy_max', 'send_v_fundamental_residual_ratio_mean', 
                     'send_v_crest_factor_mean', 'send_v_flatline_fraction_mean', 'send_i_spectral_entropy_mean', 
                     'send_i_spectral_entropy_max', 'send_i_fundamental_residual_ratio_mean', 'send_i_crest_factor_mean', 
                     'send_i_flatline_fraction_mean', 'receive_v_spectral_entropy_mean', 
                     'receive_v_spectral_entropy_max', 'receive_v_fundamental_residual_ratio_mean', 
                     'receive_v_crest_factor_mean', 'receive_v_flatline_fraction_mean', 'receive_i_spectral_entropy_mean', 
                     'receive_i_spectral_entropy_max', 'receive_i_fundamental_residual_ratio_mean', 'receive_i_crest_factor_mean', 
                     'receive_i_flatline_fraction_mean'], errors='ignore')

y = df['class_name']

X.var().sort_values(ascending=False)




send_v_rms_a_v              4.556787e+08
send_v_rms_c_v              4.510555e+08
send_v_rms_b_v              4.487309e+08
send_v1_rms_v               2.588928e+08
send_v2_rms_v               1.215947e+08
send_v0_rms_v               4.084668e+07
send_i_rms_a_a              8.178824e+05
send_i_rms_b_a              8.151528e+05
send_i_rms_c_a              8.134288e+05
send_i1_rms_a               3.114403e+05
send_i2_rms_a               1.740246e+05
send_z_magnitude_c_ohm      7.276596e+04
send_z_magnitude_a_ohm      7.270291e+04
send_z_magnitude_b_ohm      7.263535e+04
send_i0_rms_a               6.010840e+04
send_i_rms_imbalance_pct    4.342761e+03
send_v_rms_imbalance_pct    5.896697e+02
send_i2_i1_ratio            9.503513e-02
send_i0_i1_ratio            7.765606e-02
send_v2_v1_ratio            1.668311e-02
send_v0_v1_ratio            5.058841e-03
dtype: float64

### Model Testing

### Softmax Logistic Regression

In [6]:
# Logistic Regression

train = df["split"].eq("train")
validation = df["split"].eq("validation")
test = df["split"].eq("test")

X_train, y_train = X.loc[train], y.loc[train]
X_val, y_val = X.loc[validation], y.loc[validation]
X_test, y_test = X.loc[test], y.loc[test]

print(X.columns.tolist())

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

model = LogisticRegression(solver='lbfgs', max_iter=200)
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)

# Evaluate the model
for split_name, X_split, y_split in [
    ("validation", X_val_scaled, y_val),
    ("test", X_test_scaled, y_test),
    ("ood_test", scaler.transform(X.loc[df["split"].eq("ood_test")]), y.loc[df["split"].eq("ood_test")])
]:
    y_pred = model.predict(X_split)
    print(split_name, accuracy_score(y_split, y_pred))
    print(classification_report(y_split, y_pred))


['send_v_rms_a_v', 'send_v_rms_b_v', 'send_v_rms_c_v', 'send_i_rms_a_a', 'send_i_rms_b_a', 'send_i_rms_c_a', 'send_v0_rms_v', 'send_v1_rms_v', 'send_v2_rms_v', 'send_i0_rms_a', 'send_i1_rms_a', 'send_i2_rms_a', 'send_v0_v1_ratio', 'send_v2_v1_ratio', 'send_i0_i1_ratio', 'send_i2_i1_ratio', 'send_z_magnitude_a_ohm', 'send_z_magnitude_b_ohm', 'send_z_magnitude_c_ohm', 'send_v_rms_imbalance_pct', 'send_i_rms_imbalance_pct']
validation 0.9927272727272727
              precision    recall  f1-score   support

          AB       0.96      1.00      0.98       150
         ABC       1.00      1.00      1.00       150
         ABG       1.00      0.96      0.98       150
          AG       1.00      1.00      1.00       150
          BC       0.97      1.00      0.98       150
         BCG       1.00      0.97      0.98       150
          BG       1.00      1.00      1.00       150
          CA       0.99      1.00      1.00       150
         CAG       1.00      0.99      1.00       150
    

In [7]:
test_df = pd.read_csv("../data/processed/Exercise4.csv")
test_df["VA(kV)"] = test_df["VA(kV)"] * 1000
test_df["VB(kV)"] = test_df["VB(kV)"] * 1000
test_df["VC(kV)"] = test_df["VC(kV)"] * 1000

test_df.head()

,IA,IB,IC,IP,IG,VA(kV),VB(kV),VC(kV),VS(kV),V1MEM,...,ICL,IAX,IBX,ICX,IAY,IBY,ICY,IAT,IBT,ICT
0,131.0,-206.0,46.0,-12.0,-29.0,-62200.0,90800.0,-29900.0,-60.5,0.0,...,-99999.0,-99999.0,-99999.0,-99999.0,-99999.0,-99999.0,-99999.0,-99999.0,-99999.0,-99999.0
1,51.0,-211.0,123.0,-7.0,-36.0,-31200.0,92000.0,-59300.0,-33.7,0.0,...,-99999.0,-99999.0,-99999.0,-99999.0,-99999.0,-99999.0,-99999.0,-99999.0,-99999.0,-99999.0
2,-14.0,-186.0,168.0,-7.0,-32.0,4800.0,77800.0,-84100.0,7.4,0.0,...,-99999.0,-99999.0,-99999.0,-99999.0,-99999.0,-99999.0,-99999.0,-99999.0,-99999.0,-99999.0
3,-102.0,-124.0,194.0,-8.0,-32.0,39100.0,53600.0,-92600.0,36.8,0.0,...,-99999.0,-99999.0,-99999.0,-99999.0,-99999.0,-99999.0,-99999.0,-99999.0,-99999.0,-99999.0
4,-158.0,-60.0,189.0,-10.0,-28.0,70100.0,17700.0,-88400.0,71.7,-0.1,...,-99999.0,-99999.0,-99999.0,-99999.0,-99999.0,-99999.0,-99999.0,-99999.0,-99999.0,-99999.0


### Random Forest Model

In [8]:
# Random Forest Classifier
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=8,
    min_samples_leaf=1,
    max_features="sqrt",
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train, y_train)
rf_y_pred = rf_model.predict(X_test)


for split_name, X_split, y_split in [
    ("validation", X_val, y_val),
    ("test", X_test, y_test),
    ("ood_test", X.loc[df["split"].eq("ood_test")], y.loc[df["split"].eq("ood_test")])
]:
    y_pred = rf_model.predict(X_split)
    print(split_name, accuracy_score(y_split, y_pred))
    print(classification_report(y_split, y_pred))

validation 0.9993939393939394
              precision    recall  f1-score   support

          AB       1.00      1.00      1.00       150
         ABC       1.00      1.00      1.00       150
         ABG       1.00      1.00      1.00       150
          AG       1.00      1.00      1.00       150
          BC       1.00      1.00      1.00       150
         BCG       1.00      1.00      1.00       150
          BG       1.00      1.00      1.00       150
          CA       0.99      1.00      1.00       150
         CAG       1.00      0.99      1.00       150
          CG       1.00      1.00      1.00       150
     Healthy       1.00      1.00      1.00       150

    accuracy                           1.00      1650
   macro avg       1.00      1.00      1.00      1650
weighted avg       1.00      1.00      1.00      1650

test 1.0
              precision    recall  f1-score   support

          AB       1.00      1.00      1.00       150
         ABC       1.00      1.00      

### XGradient boost


In [9]:
# XGBoost Classifier
xgb_model = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=8,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

# convert labels to integers for XGBoost
label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)

# Fit the model using encoded labels
xgb_model.fit(X_train, y_train_encoded)

# Evaluate on the splits
for split_name, X_split, y_split in [
    ("validation", X_val, y_val),
    ("test", X_test, y_test),
    ("ood_test", X.loc[df["split"].eq("ood_test")], y.loc[df["split"].eq("ood_test")])
]:
    # 1. Predict the integer codes
    y_pred_encoded = xgb_model.predict(X_split)
    
    # 2. Convert the integer predictions back to original labels
    y_pred = label_encoder.inverse_transform(y_pred_encoded)
    
    # 3. Safely compare with original labels (y_split)
    print(f"--- {split_name.upper()} ---")
    print("Accuracy:", accuracy_score(y_split, y_pred))
    print(classification_report(y_split, y_pred))


--- VALIDATION ---
Accuracy: 0.9993939393939394
              precision    recall  f1-score   support

          AB       1.00      1.00      1.00       150
         ABC       1.00      1.00      1.00       150
         ABG       1.00      1.00      1.00       150
          AG       1.00      1.00      1.00       150
          BC       1.00      1.00      1.00       150
         BCG       1.00      1.00      1.00       150
          BG       1.00      1.00      1.00       150
          CA       0.99      1.00      1.00       150
         CAG       1.00      1.00      1.00       150
          CG       1.00      0.99      1.00       150
     Healthy       1.00      1.00      1.00       150

    accuracy                           1.00      1650
   macro avg       1.00      1.00      1.00      1650
weighted avg       1.00      1.00      1.00      1650

--- TEST ---
Accuracy: 1.0
              precision    recall  f1-score   support

          AB       1.00      1.00      1.00       150
   

### MLP Model

In [10]:
import torch
from torch.utils.data import DataLoader, TensorDataset

# MLP classifier

torch.manual_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Encode labels using the existing encoder
y_train_mlp = label_encoder.transform(y_train)
y_val_mlp = label_encoder.transform(y_val)
y_test_mlp = label_encoder.transform(y_test)

train_dataset = TensorDataset(
    torch.tensor(X_train_scaled, dtype=torch.float32),
    torch.tensor(y_train_mlp, dtype=torch.long)
)
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)

mlp_model = nn.Sequential(
    nn.Linear(X_train_scaled.shape[1], 128),
    nn.ReLU(),
    nn.Dropout(0.2),
    nn.Linear(128, 64),
    nn.ReLU(),
    nn.Dropout(0.2),
    nn.Linear(64, len(label_encoder.classes_))
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(mlp_model.parameters(), lr=1e-3)

# Training
for epoch in range(100):
    mlp_model.train()
    for features, labels in train_loader:
        features, labels = features.to(device), labels.to(device)

        optimizer.zero_grad()
        loss = criterion(mlp_model(features), labels)
        loss.backward()
        optimizer.step()

# Evaluation
def evaluate_mlp(X_data, y_data, split_name):
    mlp_model.eval()
    with torch.no_grad():
        features = torch.tensor(X_data, dtype=torch.float32).to(device)
        predictions = mlp_model(features).argmax(dim=1).cpu().numpy()

    predictions = label_encoder.inverse_transform(predictions)
    print(f"\n{split_name}")
    print("Accuracy:", accuracy_score(y_data, predictions))
    print(classification_report(y_data, predictions))

evaluate_mlp(X_val_scaled, y_val, "Validation")
evaluate_mlp(X_test_scaled, y_test, "Test")

ood_mask = df["split"].eq("ood_test")
evaluate_mlp(
    scaler.transform(X.loc[ood_mask]),
    y.loc[ood_mask],
    "OOD Test"
)


Validation
Accuracy: 0.9993939393939394
              precision    recall  f1-score   support

          AB       0.99      1.00      1.00       150
         ABC       1.00      1.00      1.00       150
         ABG       1.00      0.99      1.00       150
          AG       1.00      1.00      1.00       150
          BC       1.00      1.00      1.00       150
         BCG       1.00      1.00      1.00       150
          BG       1.00      1.00      1.00       150
          CA       1.00      1.00      1.00       150
         CAG       1.00      1.00      1.00       150
          CG       1.00      1.00      1.00       150
     Healthy       1.00      1.00      1.00       150

    accuracy                           1.00      1650
   macro avg       1.00      1.00      1.00      1650
weighted avg       1.00      1.00      1.00      1650


Test
Accuracy: 1.0
              precision    recall  f1-score   support

          AB       1.00      1.00      1.00       150
         ABC     

### TCN Model

In [11]:
# Temporal Convolutional Network (TCN)
class Chomp1d(nn.Module):
    def __init__(self, chomp_size):
        super().__init__()
        self.chomp_size = chomp_size

    def forward(self, x):
        return x[:, :, :-self.chomp_size] if self.chomp_size else x


class TemporalBlock(nn.Module):
    def __init__(self, in_channels, out_channels, dilation, dropout=0.2):
        super().__init__()
        padding = (3 - 1) * dilation

        self.net = nn.Sequential(
            nn.Conv1d(in_channels, out_channels, 3,
                      padding=padding, dilation=dilation),
            Chomp1d(padding),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Conv1d(out_channels, out_channels, 3,
                      padding=padding, dilation=dilation),
            Chomp1d(padding),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        self.residual = (
            nn.Conv1d(in_channels, out_channels, 1)
            if in_channels != out_channels else nn.Identity()
        )

    def forward(self, x):
        return F.relu(self.net(x) + self.residual(x))


class TCNClassifier(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.network = nn.Sequential(
            TemporalBlock(1, 32, dilation=1),
            TemporalBlock(32, 64, dilation=2),
            TemporalBlock(64, 64, dilation=4)
        )
        self.classifier = nn.Linear(64, num_classes)

    def forward(self, x):
        x = self.network(x)
        x = x.mean(dim=2)
        return self.classifier(x)


# Convert feature vectors to [samples, channels, sequence_length]
X_train_tcn = torch.tensor(
    X_train_scaled[:, None, :], dtype=torch.float32
)
X_val_tcn = torch.tensor(
    X_val_scaled[:, None, :], dtype=torch.float32
)
X_test_tcn = torch.tensor(
    X_test_scaled[:, None, :], dtype=torch.float32
)

ood_mask = df["split"].eq("ood_test")
X_ood_tcn = torch.tensor(
    scaler.transform(X.loc[ood_mask])[:, None, :],
    dtype=torch.float32
)

y_train_tcn = torch.tensor(y_train_mlp, dtype=torch.long)
y_val_tcn = torch.tensor(y_val_mlp, dtype=torch.long)
y_test_tcn = torch.tensor(y_test_mlp, dtype=torch.long)
y_ood_tcn = torch.tensor(
    label_encoder.transform(y.loc[ood_mask]),
    dtype=torch.long
)

tcn_dataset = TensorDataset(X_train_tcn, y_train_tcn)
tcn_loader = DataLoader(tcn_dataset, batch_size=128, shuffle=True)

tcn_model = TCNClassifier(
    num_classes=len(label_encoder.classes_)
).to(device)

tcn_optimizer = optim.Adam(tcn_model.parameters(), lr=1e-3)
tcn_criterion = nn.CrossEntropyLoss()

for epoch in range(100):
    tcn_model.train()

    for features, labels in tcn_loader:
        features = features.to(device)
        labels = labels.to(device)

        tcn_optimizer.zero_grad()
        predictions = tcn_model(features)
        loss = tcn_criterion(predictions, labels)
        loss.backward()
        tcn_optimizer.step()

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch + 1}/100, Loss: {loss.item():.6f}")


def evaluate_tcn(X_data, y_data, split_name):
    tcn_model.eval()

    with torch.no_grad():
        predictions = tcn_model(X_data.to(device)).argmax(dim=1)
        predictions = predictions.cpu().numpy()

    predictions = label_encoder.inverse_transform(predictions)
    actual = label_encoder.inverse_transform(y_data.numpy())

    print(f"\n{split_name}")
    print("Accuracy:", accuracy_score(actual, predictions))
    print(classification_report(actual, predictions))


evaluate_tcn(X_val_tcn, y_val_tcn, "Validation")
evaluate_tcn(X_test_tcn, y_test_tcn, "Test")
evaluate_tcn(X_ood_tcn, y_ood_tcn, "OOD Test")

Epoch 10/100, Loss: 0.096671
Epoch 20/100, Loss: 0.006613
Epoch 30/100, Loss: 0.014047
Epoch 40/100, Loss: 0.016043
Epoch 50/100, Loss: 0.002204
Epoch 60/100, Loss: 0.000474
Epoch 70/100, Loss: 0.002283
Epoch 80/100, Loss: 0.004095
Epoch 90/100, Loss: 0.007194
Epoch 100/100, Loss: 0.004869

Validation
Accuracy: 0.9993939393939394
              precision    recall  f1-score   support

          AB       0.99      1.00      1.00       150
         ABC       1.00      1.00      1.00       150
         ABG       1.00      0.99      1.00       150
          AG       1.00      1.00      1.00       150
          BC       1.00      1.00      1.00       150
         BCG       1.00      1.00      1.00       150
          BG       1.00      1.00      1.00       150
          CA       1.00      1.00      1.00       150
         CAG       1.00      1.00      1.00       150
          CG       1.00      1.00      1.00       150
     Healthy       1.00      1.00      1.00       150

    accuracy      

### Analysis

These results show logistic regression performs exceptionally well with all splits including OOD. Other models such as Random Forest struggle with unseen data as shown with poor OOD performance. Changing split to reduce easy test prediction may improve model performance.

# Alternate Split

In [12]:
df = pd.read_csv("../data/processed/stage4_features_grouped_split.csv")

target_col = "class_name"

# Drop columns that are not features
drop_cols = [
    "scenario_id",
    "split",
    "class_name",
    "operating_domain_id",
    "is_ood"
]

# Feature columns are all columns that are not in the drop_cols list
feature_cols = [c for c in df.columns if c not in drop_cols]

# Seperate the data into train, validation, test, and ood_test sets based on the "split" column
train_df = df[df["split"] == "train"]
val_df = df[df["split"] == "validation"]
test_df = df[df["split"] == "test"]
ood_df = df[df["split"] == "ood_test"]

X_train2 = train_df[feature_cols]
y_train2 = train_df[target_col]

X_val2 = val_df[feature_cols]
y_val2 = val_df[target_col]

X_test2 = test_df[feature_cols]
y_test2 = test_df[target_col]

X_ood2 = ood_df[feature_cols]
y_ood2 = ood_df[target_col]

### Softmax Logistic Regression (Alternate Split)

In [13]:
# Scale the features using StandardScaler
scaler2 = StandardScaler()

X_train2_scaled = scaler2.fit_transform(X_train2)
X_val2_scaled = scaler2.transform(X_val2)
X_test2_scaled = scaler2.transform(X_test2)
X_ood2_scaled = scaler2.transform(X_ood2)

lr_model2 = LogisticRegression(solver="lbfgs", max_iter=200)
lr_model2.fit(X_train2_scaled, y_train2)

# Evaluate the model
for split_name, X_split2, y_split2 in [
    ("validation", X_val2_scaled, y_val2),
    ("test", X_test2_scaled, y_test2),
    ("ood_test", X_ood2_scaled, y_ood2),
]:
    y_pred2 = lr_model2.predict(X_split2)
    print(split_name, accuracy_score(y_split2, y_pred2))
    print(classification_report(y_split2, y_pred2))

validation 0.9981818181818182
              precision    recall  f1-score   support

          AB       1.00      1.00      1.00       150
         ABC       1.00      1.00      1.00       150
         ABG       1.00      1.00      1.00       150
          AG       1.00      1.00      1.00       150
          BC       0.98      1.00      0.99       150
         BCG       1.00      0.98      0.99       150
          BG       1.00      1.00      1.00       150
          CA       1.00      1.00      1.00       150
         CAG       1.00      1.00      1.00       150
          CG       1.00      1.00      1.00       150
     Healthy       1.00      1.00      1.00       150

    accuracy                           1.00      1650
   macro avg       1.00      1.00      1.00      1650
weighted avg       1.00      1.00      1.00      1650

test 0.9987878787878788
              precision    recall  f1-score   support

          AB       0.99      1.00      1.00       150
         ABC       1.00 

### Random Forest Model (Alternate Split)

In [14]:
# Random Forest Classifier
# Model parameters tuned via cross-validation and grid search

rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=8,
    min_samples_leaf=1,
    max_features="sqrt",
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train2, y_train2)
rf_y_pred = rf_model.predict(X_test2)

for split_name, X_split2, y_split2 in [
    ("validation", X_val2, y_val2),
    ("test", X_test2, y_test2),
    ("ood_test", X_ood2, y_ood2),
]:
    y_pred = rf_model.predict(X_split2)
    print(split_name, accuracy_score(y_split2, y_pred))
    print(classification_report(y_split2, y_pred))

validation 1.0
              precision    recall  f1-score   support

          AB       1.00      1.00      1.00       150
         ABC       1.00      1.00      1.00       150
         ABG       1.00      1.00      1.00       150
          AG       1.00      1.00      1.00       150
          BC       1.00      1.00      1.00       150
         BCG       1.00      1.00      1.00       150
          BG       1.00      1.00      1.00       150
          CA       1.00      1.00      1.00       150
         CAG       1.00      1.00      1.00       150
          CG       1.00      1.00      1.00       150
     Healthy       1.00      1.00      1.00       150

    accuracy                           1.00      1650
   macro avg       1.00      1.00      1.00      1650
weighted avg       1.00      1.00      1.00      1650

test 1.0
              precision    recall  f1-score   support

          AB       1.00      1.00      1.00       150
         ABC       1.00      1.00      1.00       150


### Conclusion

The alternate grouped split reveals that the Random Forest and potentially other models understand validation and test splits thoroughly. These models' perfect scores may not be a sign of data leakage or overfitting, but rather a reflection of weak test values and strong OOD values. Regardless, Softmax Logistic Regression (SLR) provides the best results across all splits. OOD predictions are consistent with SLR, meaning this model understands unseen data, unlike the tree and neural network models. Overall, based on these test results, Softmax Logistic Regression with the selected features is the most robust and efficient model tested, suggesting it is sufficient for this classification task.